# 09. Putting it together

To close the track, an end-to-end experiment: **plan** the sample size (`power_analysis`), **run** it with automatic diagnostics (`ExperimentPipeline`), and **report** it (`ExperimentReport`).

## 1. Plan: how many units?

`power_analysis` solves for one of (n, MDE, power) given the other two. Here we ask for the `n` needed to detect an effect of `0.3` with 80% power.

In [ ]:
from skxperiments.design.power import power_analysis

plan = power_analysis(mde=0.3, power=0.8, std=1.0, alpha=0.05)
print(f"required n: {plan.n_total} (about {plan.n_treated} per group)")

## 2. Run: pipeline with automatic diagnostics

`ExperimentPipeline` runs the diagnostics (SRM by default) and the inference in one step, bundling everything into a `PipelineResult`.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI
from skxperiments.pipeline import ExperimentPipeline

n = plan.n_total
rng = np.random.default_rng(0)
y0 = rng.normal(size=n)
tau = 0.3
y1 = y0 + tau
df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=0)
a = design.randomize(df)
t = a.data_[a.treatment_col_].to_numpy()
data = a.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
a = CRDAssignment(data=data, treatment_col=a.treatment_col_, design=design, seed=0)

pr = ExperimentPipeline(
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="y")),
    diagnostics=[SRMTest()],
).run(a)
print(f"ATE={pr.results.ate:.3f}, p={pr.results.p_value:.4f}, flag={pr.flagged}")

## 3. Report

`ExperimentReport` builds a self-contained HTML page with the results table, the diagnostics, and the embedded plots. Here we render it inline; in production, use `report.save("report.html")`.

In [ ]:
from IPython.display import HTML

from skxperiments.reporting import ExperimentReport

report = ExperimentReport(pr, title="Experiment report")
HTML(report.to_html())

## End of the track

You went through design, estimation, three forms of inference, variance reduction, balance, blocking, factorial, multiple testing, diagnostics, and reporting. That is the library's full flow.

This is the **for_starters** track (simulated data, didactic). Soon, a **real-data** track will show the library on real-world cases.